In [3]:
import numpy as np
import tensorflow as tf
import pandas as pd
from bkm10 import BKM10
import time

In [19]:
fn = 'finalised pseudodata.csv'
df_all = pd.read_csv(fn)

ranges = [[-30., 30.], 
            [-30., 30.], 
            [-100., 100.], 
            [-400., 400.], 
            [-20., 20.],
            [-30., 30.],
            [-100., 100.],
            [-400., 400.]]

In [ ]:
def get_sig(plus, mins, phi, cffs):

    splus = plus.calculate_cross_section(phi, cffs)
    smins = mins.calculate_cross_section(phi, cffs)

    dsig = 0.5*(splus+smins)
    delsig = 0.5*(splus-smins)

    return dsig, delsig

def MSE(y1, y1_true, y2, y2_true):
    return tf.reduce_sum(tf.abs(y1-y1_true)**2+tf.abs(y2-y2_true)**2, axis=1)

In [51]:
def run(set_num, restarts, epochs):
    
    df = df_all[df_all['set']==set_num].copy()

    plus = BKM10(*df[['k', 'Q2', 'xB', 't']].values[0], helc=1)
    mins = BKM10(*df[['k', 'Q2', 'xB', 't']].values[0], helc=-1)
    names = ['ReH','ReHt','ReE','ReEt','ImH','ImHt','ImE','ImEt']
    cffs = df[names].to_numpy()[0]

    phi = np.linspace(min(df['phi']), max(df['phi']), 20)
    phi = np.pi-np.deg2rad(phi)
    phi = tf.convert_to_tensor(phi, dtype=tf.float32)

    true_cffs = tf.convert_to_tensor(cffs, dtype=tf.float32)

    dsig_true, delsig_true = get_sig(plus, mins, phi, tf.expand_dims(true_cffs, axis=0))

    free_cff_indices = tf.constant([0,1,4,5], dtype=tf.int32)

    range_mins = tf.constant([ranges[i][0] for i in free_cff_indices], dtype=tf.float32)
    range_maxs = tf.constant([ranges[i][1] for i in free_cff_indices], dtype=tf.float32)

    free_cffs = tf.Variable(
        tf.random.uniform([restarts, len(free_cff_indices)], dtype=tf.float32) * (range_maxs - range_mins) + range_mins)

    row_idx = tf.repeat(tf.range(restarts), len(free_cff_indices))        # [N*4]
    col_idx = tf.tile(free_cff_indices, [restarts])                        # [N*4]
    indices = tf.stack([row_idx, col_idx], axis=1) 
    base = tf.tile(true_cffs[tf.newaxis, :], [restarts, 1])

    optimizer = tf.keras.optimizers.Adam(learning_rate=1., epsilon=1e-12)

    @tf.function
    def train_step():
        with tf.GradientTape() as tape:
            cffs = tf.tensor_scatter_nd_update(base, indices, tf.reshape(free_cffs, [-1]))
            dsig_pred, delsig_pred = get_sig(plus, mins, phi, cffs)
            losses = MSE(dsig_pred, dsig_true, delsig_pred, delsig_true)
        gradient = tape.gradient(losses, free_cffs)
        optimizer.apply_gradients([(gradient, free_cffs)])
        return losses

    start = time.perf_counter()
    for i in range(epochs):
        loss = train_step()
        # if (i % 100 == 0):
        #     print(i, loss.numpy(), free_val.numpy())
    # print(loss.numpy(), free_cffs.numpy())

    pred_cffs = free_cffs.numpy()[np.argmin(loss.numpy())]
    true_cffs_selected = [float(cffs[i]) for i in free_cff_indices]
    err = abs((true_cffs_selected-pred_cffs)*100/true_cffs_selected)

    print(f'CFFs = {[names[i] for i in free_cff_indices]}')
    print(f'Pred = {pred_cffs}, Min Loss= {np.min(loss.numpy())}')
    print(f'True = {true_cffs_selected}')
    print(f'Err = {err}%')
    end = time.perf_counter()
    print(f"Elapsed time: {end - start:.6f} seconds")

In [53]:
run(set_num= 4, restarts=10000, epochs= 1000)

CFFs = ['ReH', 'ReHt', 'ImH', 'ImHt']
Pred = [-2.514828   1.347399   3.203218   1.4988481], Min Loss= 6.123573870198129e-15
True = [-2.51484, 1.34742, 3.20275, 1.49975]
Err = [0.00047849 0.0015588  0.01461192 0.06013798]%
Elapsed time: 12.503832 seconds
